In [1]:
import os
import sqlite3
from pathlib import Path

import pandas as pd
import requests
from typing import Optional
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
API_KEY = os.getenv("DATA_GOV_API_KEY")

RESOURCE_ID = os.getenv("RESOURCE_ID")
API_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

DB_PATH = Path("../database/mandi_prices.db")

SUPPORTED_COMMODITIES = {
    "Brinjal",
    "Cauliflower",
    "Green Chilli",
    "Onion",
    "Potato",
    "Tomato",
}

In [4]:
def fetch_current_prices(
    state: str,
    commodity: str,
    limit: int = 100,
    market: Optional[str] = None,
    district: Optional[str] = None,
    timeout: int = 30,
) -> Optional[pd.DataFrame]:
    """
    Hits the data.gov.in Agmarknet resource for the most recent record
    matching state/commodity(/market)(/district). Returns None on any
    failure so the caller can fall back to the last known CSV value.
    """
    params = {
        "api-key": API_KEY,
        "format": "json",
        "limit": limit,
        "filters[state.keyword]": state,
        "filters[commodity]": commodity,
    }
    if market:
        params["filters[market]"] = market
    if district:
        params["filters[district]"] = district

    headers = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(API_URL, params=params, timeout=timeout, headers=headers)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "ok":
            raise RuntimeError(f"data.gov.in returned: {data}")
        records = data.get("records") or []
        return pd.DataFrame(records) if records else pd.DataFrame()
    except requests.RequestException as e:
        print(f"[agmarknet_api] API error: {e}")
        return None

In [5]:
fetch_current_prices(state="Andhra Pradesh", commodity="Tomato", limit=50)

,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price
0,Andhra Pradesh,Annamayya,Somala APMC,Tomato,Tomato,FAQ,09/07/2026,1000,1600,1300
1,Andhra Pradesh,Chittor,Palamaner APMC,Tomato,Tomato,FAQ,09/07/2026,920,1290,1050
2,Andhra Pradesh,Annamayya,Punganur APMC,Tomato,Hybrid,FAQ,09/07/2026,900,1300,1100
3,Andhra Pradesh,Annamayya,Guramkonda Sub-yard of AMC Valmikipuram,Tomato,Hybrid,FAQ,09/07/2026,920,1280,1080
4,Andhra Pradesh,Annamayya,Valmikipuram APMC,Tomato,Hybrid,FAQ,09/07/2026,920,1280,1080
5,Andhra Pradesh,Annamayya,B.Kothakota H/Q Angallu,Tomato,Local,FAQ,09/07/2026,1000,1700,1300
6,Andhra Pradesh,Annamayya,Madanapalli APMC,Tomato,Local,FAQ,09/07/2026,1500,1800,1700
7,Andhra Pradesh,Annamayya,Mulakalacheruvu APMC,Tomato,Local,FAQ,09/07/2026,900,1600,1300
8,Andhra Pradesh,Chittor,V. Kota Sub-yard of AMC Palamaneru,Tomato,Other,FAQ,09/07/2026,800,1130,1000
9,Andhra Pradesh,Annamayya,Kalikiri APMC,Tomato,Hybrid,FAQ,09/07/2026,900,1500,1200


In [6]:
fetch_current_prices(state= "Andhra Pradesh", district="Annamayya", commodity="Tomato", limit=100) 

,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price
0,Andhra Pradesh,Annamayya,Somala APMC,Tomato,Tomato,FAQ,09/07/2026,1000,1600,1300
1,Andhra Pradesh,Annamayya,Punganur APMC,Tomato,Hybrid,FAQ,09/07/2026,900,1300,1100
2,Andhra Pradesh,Annamayya,Guramkonda Sub-yard of AMC Valmikipuram,Tomato,Hybrid,FAQ,09/07/2026,920,1280,1080
3,Andhra Pradesh,Annamayya,Valmikipuram APMC,Tomato,Hybrid,FAQ,09/07/2026,920,1280,1080
4,Andhra Pradesh,Annamayya,B.Kothakota H/Q Angallu,Tomato,Local,FAQ,09/07/2026,1000,1700,1300
5,Andhra Pradesh,Annamayya,Madanapalli APMC,Tomato,Local,FAQ,09/07/2026,1500,1800,1700
6,Andhra Pradesh,Annamayya,Mulakalacheruvu APMC,Tomato,Local,FAQ,09/07/2026,900,1600,1300
7,Andhra Pradesh,Annamayya,Kalikiri APMC,Tomato,Hybrid,FAQ,09/07/2026,900,1500,1200


In [7]:
# ── 2. Convert API records to your database schema ───────────────────────────

def normalize_api_data(raw_df):
    """
    Convert API data to your existing database columns.

    Fields unavailable in the API are deliberately None:
    commodity_group, arrival_quantity, arrival_unit.
    """
    if raw_df.empty:
        return pd.DataFrame()

    df = raw_df.copy()

    # API date is DD/MM/YYYY -> database date is YYYY-MM-DD
    df["arrival_date"] = pd.to_datetime(
        df["arrival_date"],
        format="%d/%m/%Y",
        errors="coerce",
    ).dt.strftime("%Y-%m-%d")

    # API price fields can arrive as strings
    for col in ["min_price", "max_price", "modal_price"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Missing in this API
    df["commodity_group"] = 'Vegetables'
    df["price_unit"] = "Rs./Quintal"
    df["arrival_quantity"] = None
    df["arrival_unit"] = None

    # Keep only the commodities already supported by your database
    df = df[df["commodity"].isin(SUPPORTED_COMMODITIES)].copy()

    # Remove unusable rows
    df = df.dropna(
        subset=["state", "market", "commodity", "arrival_date", "modal_price"]
    )

    # Prevent duplicate rows returned in one API response
    key_columns = [
        "arrival_date",
        "state",
        "district",
        "market",
        "commodity",
        "variety",
        "grade",
    ]
    df = df.drop_duplicates(subset=key_columns, keep="last")

    db_columns = [
        "state",
        "district",
        "market",
        "commodity_group",
        "commodity",
        "variety",
        "grade",
        "min_price",
        "max_price",
        "modal_price",
        "price_unit",
        "arrival_quantity",
        "arrival_unit",
        "arrival_date",
    ]

    return df[db_columns]

In [8]:
# ── 3. Insert/update without deleting AGMARKNET arrival values ───────────────

def upsert_prices_to_db(df):
    """
    Insert API rows into the same table.

    If a record already exists for the same market/crop/date:
    - price fields are updated
    - commodity_group, arrival_quantity, and arrival_unit are preserved
    """
    if df.empty:
        print("No supported mandi records found in the API response.")
        return 0

    rows = df.where(pd.notna(df), None).to_dict(orient="records")

    conn = sqlite3.connect(DB_PATH)

    conn.executemany(
        """
        INSERT INTO mandi_market_data (
            state, district, market, commodity_group, commodity,
            variety, grade, min_price, max_price, modal_price,
            price_unit, arrival_quantity, arrival_unit, arrival_date
        )
        VALUES (
            :state, :district, :market, :commodity_group, :commodity,
            :variety, :grade, :min_price, :max_price, :modal_price,
            :price_unit, :arrival_quantity, :arrival_unit, :arrival_date
        )
        ON CONFLICT(
            arrival_date, state, district, market,
            commodity, variety, grade
        )
        DO UPDATE SET
            min_price = excluded.min_price,
            max_price = excluded.max_price,
            modal_price = excluded.modal_price,
            price_unit = excluded.price_unit
        """,
        rows,
    )

    conn.commit()
    conn.close()

    return len(rows)

In [9]:
# ── 4. Run one complete daily sync ───────────────────────────────────────────

def sync_daily_mandi_prices(
    state: str,
    commodity: str,
    limit: int = 100,
    market: Optional[str] = None,
    district: Optional[str] = None,
    timeout: int = 30,
    ):
    raw_df = fetch_current_prices(state=state, commodity=commodity, limit=limit, market=market, district=district)

    if raw_df is None or raw_df.empty:
        print(f"No API data found for {commodity} in {state}.")
        return

    clean_df = normalize_api_data(raw_df)
    synced_rows = upsert_prices_to_db(clean_df)

    print(f"Raw API records fetched: {len(raw_df)}")
    print(f"Rows synced for your six commodities: {synced_rows}")




In [10]:
def sync_all_commodities():
    commodity = ["Tomato", "Onion", "Potato", "Cauliflower", "Brinjal", "Green Chilli"]
    for item in commodity:
        print(f"Syncing daily mandi prices for {item} in Andhra Pradesh...")
        sync_daily_mandi_prices(state="Andhra Pradesh", commodity=item, limit=100)
        print("\n" + "=" * 50 + "\n")

In [11]:
sync_all_commodities()

Syncing daily mandi prices for Tomato in Andhra Pradesh...
Raw API records fetched: 10
Rows synced for your six commodities: 10


Syncing daily mandi prices for Onion in Andhra Pradesh...
No API data found for Onion in Andhra Pradesh.


Syncing daily mandi prices for Potato in Andhra Pradesh...
Raw API records fetched: 1
Rows synced for your six commodities: 1


Syncing daily mandi prices for Cauliflower in Andhra Pradesh...
Raw API records fetched: 1
Rows synced for your six commodities: 1


Syncing daily mandi prices for Brinjal in Andhra Pradesh...
Raw API records fetched: 1
Rows synced for your six commodities: 1


Syncing daily mandi prices for Green Chilli in Andhra Pradesh...
Raw API records fetched: 1
Rows synced for your six commodities: 1




In [ ]:
import sqlite3
import pandas as pd


def fetch_historical_prices(
    commodity: str,
    market: str | None = None,
    variety: str | None = None,
    state: str | None = None,
    district: str | None = None,
    days: int = 7,
    db_path: str = DB_PATH,
) -> pd.DataFrame:
    """
    Fetch recent mandi-price records with optional location and variety filters.

    If market/variety are omitted, returns records across all matching
    markets/varieties. `days` means the latest N distinct reporting dates,
    not merely N rows.
    """

    query = """
    SELECT
        arrival_date,
        state,
        district,
        market,
        commodity_group,
        commodity,
        variety,
        grade,
        min_price,
        max_price,
        modal_price,
        price_unit,
        arrival_quantity,
        arrival_unit
    FROM mandi_market_data
    WHERE commodity = ?
    """

    params = [commodity]

    if market is not None:
        query += " AND market = ?"
        params.append(market)

    if variety is not None:
        query += " AND variety = ?"
        params.append(variety)

    if state is not None:
        query += " AND state = ?"
        params.append(state)

    if district is not None:
        query += " AND district = ?"
        params.append(district)

    # Select latest N DISTINCT dates first, then return all matching rows
    query += """
    AND arrival_date IN (
        SELECT DISTINCT arrival_date
        FROM mandi_market_data
        WHERE commodity = ?
    """

    date_params = [commodity]

    if market is not None:
        query += " AND market = ?"
        date_params.append(market)

    if variety is not None:
        query += " AND variety = ?"
        date_params.append(variety)

    if state is not None:
        query += " AND state = ?"
        date_params.append(state)

    if district is not None:
        query += " AND district = ?"
        date_params.append(district)

    query += """
        ORDER BY arrival_date DESC
        LIMIT ?
    )
    ORDER BY arrival_date ASC, market ASC, variety ASC
    """

    params.extend(date_params)
    params.append(days)

    with sqlite3.connect(db_path) as conn:
        history = pd.read_sql_query(query, conn, params=params)

    history["arrival_date"] = pd.to_datetime(history["arrival_date"])
    return history

In [44]:
history = fetch_historical_prices(
    commodity="Brinjal",
    state="Andhra Pradesh",
    market="Palamaner APMC",
    days=7
)

In [45]:
history

,arrival_date,state,district,market,commodity_group,commodity,variety,grade,min_price,max_price,modal_price,price_unit,arrival_quantity,arrival_unit
0,2026-07-02,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,5000.0,4000.0,Rs./Quintal,1.7,Metric Tonnes
1,2026-07-04,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,4000.0,3500.0,Rs./Quintal,2.1,Metric Tonnes
2,2026-07-05,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,4000.0,3500.0,Rs./Quintal,2.2,Metric Tonnes
3,2026-07-06,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,3000.0,4000.0,3500.0,Rs./Quintal,2.4,Metric Tonnes
4,2026-07-07,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,2500.0,3500.0,3000.0,Rs./Quintal,2.1,Metric Tonnes
5,2026-07-08,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,2700.0,4000.0,3500.0,Rs./Quintal,1.9,Metric Tonnes
6,2026-07-09,Andhra Pradesh,Chittor,Palamaner APMC,Vegetables,Brinjal,Brinjal,FAQ,2500.0,4500.0,3500.0,Rs./Quintal,NaN,None
